In [1]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_train = pd.read_csv("../data/ml_ready/train.csv")
X_train = df_train.drop("next_monthly_revenue", axis=1)
y_train = df_train["next_monthly_revenue"]

df_valid = pd.read_csv("../data/ml_ready/valid.csv")
X_valid = df_valid.drop("next_monthly_revenue", axis=1)
y_valid = df_valid["next_monthly_revenue"]

In [3]:
loaded_scaler = joblib.load("../models/scaler.joblib")
loaded_models = joblib.load("../models/random_forest_model.joblib")

In [4]:
X_train_scaled = loaded_scaler.transform(X_train)
X_valid_scaled = loaded_scaler.transform(X_valid)

In [5]:
loaded_models.fit(X_train_scaled, y_train)
rf_y_pred = loaded_models.predict(X_valid_scaled)

In [6]:
rf_y_pred.shape

(21762,)

In [7]:
abs = (y_valid - rf_y_pred).abs()

In [8]:
print("MAE", mean_absolute_error(y_valid, rf_y_pred))
print("RMSE", np.sqrt(mean_squared_error(y_valid, rf_y_pred)))
print("R2 Score", r2_score(y_valid, rf_y_pred))
print("Median Absolute Error", abs.median())
print("75th Percentile", abs.quantile(0.75))
print("90th Percentile", abs.quantile(0.9))
print("95th Percentile", abs.quantile(0.95))
print("99th Percentile", abs.quantile(0.99))
print("Maximum Absolute Error", abs.max())

MAE 18440.945410991884
RMSE 56273.07413100883
R2 Score 0.8110728926463513
Median Absolute Error 5157.227588395615
75th Percentile 13578.851618240189
90th Percentile 34782.17953428745
95th Percentile 74687.35487954337
99th Percentile 255896.409824874
Maximum Absolute Error 1425689.2383501977


In [9]:
df_valid.columns

Index(['month_number', 'monthly_revenue', 'revenue_per_customer', 'sin_month',
       'cos_month', 'monthly_quantity', 'lag_1_revenue', 'lag_2_revenue',
       'lag_3_revenue', 'lag_12_revenue', 'lag_1_quantity', 'lag_2_quantity',
       'lag_3_quantity', 'lag_12_quantity', 'lag_1_transactions',
       'lag_2_transactions', 'lag_3_transactions', 'lag_12_transactions',
       'moving_avg_3_month', 'moving_avg_6_month', 'moving_avg_12_month',
       'month_over_month', 'year_on_year', 'moving_stdev_3_month',
       'moving_stdev_6_month', 'momentum', 'total_profit', 'average_profit',
       'average_discount_pct', 'maximum_discount', 'total_discount_amount',
       'pct_transactions_discount_gt_10', 'average_selling_price',
       'minimum_selling_price', 'maximum_selling_price', 'price_range',
       'non_discounted_revenue', 'discounted_revenue', 'next_monthly_revenue'],
      dtype='str')

In [55]:
df_valid.columns

Index(['month_number', 'monthly_revenue', 'revenue_per_customer', 'sin_month',
       'cos_month', 'monthly_quantity', 'lag_1_revenue', 'lag_2_revenue',
       'lag_3_revenue', 'lag_12_revenue', 'lag_1_quantity', 'lag_2_quantity',
       'lag_3_quantity', 'lag_12_quantity', 'lag_1_transactions',
       'lag_2_transactions', 'lag_3_transactions', 'lag_12_transactions',
       'moving_avg_3_month', 'moving_avg_6_month', 'moving_avg_12_month',
       'month_over_month', 'year_on_year', 'moving_stdev_3_month',
       'moving_stdev_6_month', 'momentum', 'total_profit', 'average_profit',
       'average_discount_pct', 'maximum_discount', 'total_discount_amount',
       'pct_transactions_discount_gt_10', 'average_selling_price',
       'minimum_selling_price', 'maximum_selling_price', 'price_range',
       'non_discounted_revenue', 'discounted_revenue', 'next_monthly_revenue'],
      dtype='str')

In [56]:
error_df = pd.DataFrame({
        "Actual": y_valid,
        "Predicted": rf_y_pred,
        "monthly_revenue": df_valid["monthly_revenue"],
        "moving_avg_12": df_valid["moving_avg_12_month"],
        "Absolute Error": (y_valid - rf_y_pred).abs(),
        "Signed Error": (y_valid - rf_y_pred),
        "Percentage Error": ((y_valid - rf_y_pred)/y_valid)*100,
        "Signed Percentage Error": ((y_valid - rf_y_pred).abs()/y_valid)*100,
        "month_number": df_valid["month_number"]
})

In [57]:
pd.options.display.float_format = '{:.2f}'.format
error_df.sort_values(by="Absolute Error", ascending=False).head(20)

,Actual,Predicted,monthly_revenue,moving_avg_12,Absolute Error,Signed Error,Percentage Error,Signed Percentage Error,month_number
19161,2429583.75,1003894.51,1198642.50,965772.50,1425689.24,1425689.24,58.68,58.68,1
19179,2472525.00,1189097.25,856845.00,1200375.00,1283427.75,1283427.75,51.91,51.91,7
19133,2193048.00,980569.42,105732.00,970794.00,1212478.58,1212478.58,55.29,55.29,9
19182,2050785.00,958538.06,1782000.00,1334767.50,1092246.94,1092246.94,53.26,53.26,10
19119,1621158.00,683637.18,964656.00,867999.00,937520.82,937520.82,57.83,57.83,7
19135,432432.00,1275353.18,749628.00,1004256.00,842921.18,-842921.18,-194.93,194.93,11
19132,105732.00,936961.82,1533708.00,1033560.00,831229.82,-831229.82,-786.17,786.17,8
20465,1821204.00,1049362.12,1050084.00,1158475.50,771841.88,771841.88,42.38,42.38,7
19899,1542240.00,778447.98,950670.00,834592.50,763792.02,763792.02,49.52,49.52,10
20455,1552036.50,805277.84,86782.50,667007.25,746758.66,746758.66,48.11,48.11,9


In [58]:
error_df.shape

(21762, 9)

In [59]:
df = pd.read_csv("../data/processed/ml_dataset.csv")

In [60]:
df.isna().sum()

product_id                             0
product_name                           0
year                                   0
month                                  0
month_number                           0
monthly_revenue                        0
unique_customers                     191
units_per_customer                   191
revenue_per_customer                 191
next_monthly_revenue                1820
is_quarter_end                         0
is_year_end                            0
trend_year                             0
sin_month                              0
cos_month                              0
monthly_quantity                       0
monthly_transactions                   0
lag_1_revenue                       1820
lag_2_revenue                       3640
lag_3_revenue                       5460
lag_12_revenue                     21840
lag_1_quantity                      1820
lag_2_quantity                      3640
lag_3_quantity                      5460
lag_12_quantity 

In [61]:
for col in df.columns:
    if df[col].isna().sum() == 191:
        df = df.fillna({col: 0})

In [63]:
df = df.dropna(axis=0)
df.shape

(85244, 54)

In [64]:
validated_df = df[df["year"] == 2024]

In [65]:
validated_df.shape

(21762, 54)

In [66]:
validated_df_subset = validated_df[['product_id', 'product_name']]

In [67]:
resultant_df = pd.concat([validated_df_subset.reset_index(drop=True), error_df.reset_index(drop=True)], axis=1)

In [70]:
top_20_errors = resultant_df.sort_values(by="Absolute Error", ascending=False).head(20)

In [71]:
top_20_errors["month_number"].value_counts()

month_number
7     3
9     3
10    3
8     3
11    2
4     2
1     1
3     1
12    1
5     1
Name: count, dtype: int64

In [74]:
top_20_errors[["product_id", "product_name"]].value_counts()

product_id  product_name                 
P1600       Laptop Standard 512GB            4
P1712       Air Conditioner Premium 512GB    4
P1604       Laptop Premium 512GB             2
P1599       Laptop Standard 256GB            2
P1711       Air Conditioner Premium 256GB    2
P1603       Laptop Premium 256GB             1
P1664       Television Premium 512GB         1
P1710       Air Conditioner Premium 128GB    1
P1602       Laptop Premium 128GB             1
P1675       Refrigerator Premium 256GB       1
P1611       Tablet Standard 256GB            1
Name: count, dtype: int64

In [76]:
top_20_errors.columns

Index(['product_id', 'product_name', 'Actual', 'Predicted', 'monthly_revenue',
       'moving_avg_12', 'Absolute Error', 'Signed Error', 'Percentage Error',
       'Signed Percentage Error', 'month_number'],
      dtype='str')

In [78]:
top_20_errors

,product_id,product_name,Actual,Predicted,monthly_revenue,moving_avg_12,Absolute Error,Signed Error,Percentage Error,Signed Percentage Error,month_number
19161,P1603,Laptop Premium 256GB,2429583.75,1003894.51,1198642.50,965772.50,1425689.24,1425689.24,58.68,58.68,1
19179,P1604,Laptop Premium 512GB,2472525.00,1189097.25,856845.00,1200375.00,1283427.75,1283427.75,51.91,51.91,7
19133,P1600,Laptop Standard 512GB,2193048.00,980569.42,105732.00,970794.00,1212478.58,1212478.58,55.29,55.29,9
19182,P1604,Laptop Premium 512GB,2050785.00,958538.06,1782000.00,1334767.50,1092246.94,1092246.94,53.26,53.26,10
19119,P1599,Laptop Standard 256GB,1621158.00,683637.18,964656.00,867999.00,937520.82,937520.82,57.83,57.83,7
19135,P1600,Laptop Standard 512GB,432432.00,1275353.18,749628.00,1004256.00,842921.18,-842921.18,-194.93,194.93,11
19132,P1600,Laptop Standard 512GB,105732.00,936961.82,1533708.00,1033560.00,831229.82,-831229.82,-786.17,786.17,8
20465,P1712,Air Conditioner Premium 512GB,1821204.00,1049362.12,1050084.00,1158475.50,771841.88,771841.88,42.38,42.38,7
19899,P1664,Television Premium 512GB,1542240.00,778447.98,950670.00,834592.50,763792.02,763792.02,49.52,49.52,10
20455,P1711,Air Conditioner Premium 256GB,1552036.50,805277.84,86782.50,667007.25,746758.66,746758.66,48.11,48.11,9
